In [1]:
import numpy as np

In [17]:
nslots = 12
ntrials = 10000000

def addcol(slots, afterlast=False):
    slots = np.column_stack((slots, -np.ones(ntrials, dtype=int)))
    chix = np.arange(slots.shape[0])
    if afterlast:
        chix = chix[np.all(slots != nslots, axis=1)]
    while chix.shape[0]:
        if afterlast:
            slots[chix, -1] = np.random.randint(np.max(slots[chix], axis=1)+1, nslots+1, chix.shape[0])
        else:
            slots[chix, -1] = np.random.randint(1, nslots+1, chix.shape[0])
        chix = chix[np.any(slots[chix, -1:] == slots[chix, :-1], axis=1)]
    return slots

# First 3
slots = np.random.randint(1, nslots+1, (ntrials, 1))
for i in range(2):
    slots = addcol(slots)
while np.any(slots[:, -1] != -1):
    slots = addcol(slots, True)
n_rides = np.sum(slots != -1, axis=1)

print(np.column_stack(np.unique(n_rides, return_counts=True)))
print()
print(np.mean(n_rides))

[[      3 2502378]
 [      4 3798388]
 [      5 2502496]
 [      6  938447]
 [      7  220557]
 [      8   34166]
 [      9    3371]
 [     10     188]
 [     11       9]]

4.2693393


In [18]:
def get_exp_nrides(lslot):
    if lslot == nslots:
        return {0: 1.}
    else:
        results = dict()
        sl_prob = 1 / (nslots-lslot)
        for new_lslot in range(lslot+1, nslots+1):
            for k, v in get_exp_nrides(new_lslot).items():
                if (k+1) not in results.keys():
                    results[k+1] = 0.
                results[k+1] += v*sl_prob
        return results

# Latest slot probability after initial allocation
lslot = np.arange(3, 13)
lslot_prob = lslot / nslots * (lslot-1) / (nslots-1) * (lslot-2) / (nslots-2)
lslot_prob[1:] -= lslot_prob[:-1]
lslot_prob = dict(zip(lslot.tolist(), lslot_prob.tolist()))

# Additional ride probs
results = dict()
for k1, v1 in lslot_prob.items():
    for k2, v2 in get_exp_nrides(k1).items():
        if (k2+3) not in results.keys():
            results[k2+3] = 0.
        results[k2+3] += v2*v1

for k in sorted(results.keys()):
    print(k, '%.8f' % results[k])
print()
print(sum([k * results[k] for k in results.keys()]))

3 0.25000000
4 0.37996934
5 0.25024937
6 0.09387546
7 0.02213463
8 0.00340620
9 0.00034249
10 0.00002172
11 0.00000079
12 0.00000001

4.269877344877345
